# Dota 2: leak-free prediction of a Radiant win

Этот ноутбук — проверяемый интерфейс к пакету `dota_predictor`. Вся логика контракта данных, feature engineering и nested CV находится в обычных Python-модулях и покрыта тестами.

> Исторический ROC-AUC около 0.84 отозван после обнаружения post-match признаков. Результат ниже считается валидным только после повторного запуска нового пайплайна на исходном датасете.

## 1. Окружение и пути

Разместите данные по инструкции из `DATA.md`. JSONL используется только для извлечения выбранных героев; итоговые поля полного матча не присоединяются.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.metrics import RocCurveDisplay

from dota_predictor.data import (
    attach_pregame_metadata,
    load_pregame_metadata_jsonl,
    load_training_data,
)
from dota_predictor.evaluation import nested_cross_validate
from dota_predictor.features import add_team_aggregates
from dota_predictor.modeling import build_catboost_pipeline, build_logistic_pipeline

FEATURES_PATH = Path("train_features.csv")
TARGETS_PATH = Path("train_targets.csv")
MATCHES_PATH = Path("DOTA 2/train_matches.jsonl")

In [ ]:
required_files = [FEATURES_PATH, TARGETS_PATH, MATCHES_PATH]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f"Добавьте файлы по DATA.md: {missing_files}")

## 2. Строгая загрузка данных

Loader проверяет уникальность и полное совпадение `match_id_hash`, бинарный target и отсутствие известных future/outcome колонок.

In [ ]:
features, target = load_training_data(FEATURES_PATH, TARGETS_PATH)
metadata = load_pregame_metadata_jsonl(MATCHES_PATH, fields=("hero_name",))
features = attach_pregame_metadata(features, metadata)
features = add_team_aggregates(features)

pd.DataFrame(
    {"value": [len(features), features.shape[1], int(target.sum()), int((1 - target).sum())]},
    index=["matches", "features", "radiant_wins", "dire_wins"],
)

## 3. Что делает hero encoder

Внутри каждого train-fold `fit_transform` вычитает исход текущего матча из статистики каждого героя. Для validation используется mapping только из train-fold. Редкие и неизвестные герои сглаживаются к prior.

In [ ]:
hero_columns = [column for column in features if column.endswith("_hero_name")]
features[hero_columns].head()

## 4. Nested cross-validation

Outer folds измеряют качество, inner folds выбирают параметры. Один внешний validation-fold не используется ни для target encoding, ни для настройки модели. Для быстрого локального smoke-run установите `OUTER_SPLITS = 3`, `INNER_SPLITS = 2`.

In [ ]:
RUN_EVALUATION = False
MODEL = "logistic"  # logistic | catboost
OUTER_SPLITS = 5
INNER_SPLITS = 3

if MODEL == "catboost":
    estimator = build_catboost_pipeline()
    param_grid = {
        "model__depth": [5, 7],
        "model__learning_rate": [0.03, 0.08],
    }
else:
    estimator = build_logistic_pipeline()
    param_grid = {"model__C": [0.1, 1.0, 10.0]}

In [ ]:
result = None
if RUN_EVALUATION:
    result = nested_cross_validate(
        estimator,
        param_grid,
        features,
        target,
        outer_splits=OUTER_SPLITS,
        inner_splits=INNER_SPLITS,
    )
    display(pd.DataFrame(result.to_dict()["folds"]))
    display(result.to_dict())
else:
    print("Установите RUN_EVALUATION = True для запуска честной оценки.")

In [ ]:
if result is not None:
    RocCurveDisplay.from_predictions(target, result.oof_predictions, name=MODEL)
    plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
    plt.title("Outer-fold OOF ROC curve")
    plt.grid(alpha=0.2)
    plt.show()

## 5. Интерпретация результата

В отчёт следует переносить только `oof_roc_auc`, mean/std внешних фолдов и параметры, выбранные внутри каждого фолда. Inner-CV best score не является итоговой метрикой.

Перед production-сценарием дополнительно нужны time-based holdout по более позднему патчу, calibration curve, проверка drift и фиксация точного snapshot timestamp.